# Proyecto 1 · Diagnóstico del estado inicial de los datos

**CC3084 – Data Science · Semestre II 2026**

- **Fuente:** Búsqueda de establecimientos del MINEDUC — http://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/
- **Filtro aplicado en la fuente:** NIVEL ESCOLAR = DIVERSIFICADO (todo el país)
- **Fecha de extracción:** 17/07/2026
- **Archivo crudo:** `data/raw/MINEDUC_establecimientos_diversificado.csv` (no se modifica; es la fuente de verdad)

Este cuaderno diagnostica el estado del conjunto de datos **antes** de cualquier limpieza.
Todo hallazgo se respalda con tablas generadas por código.

**Sección 1 — Descripción global** del conjunto de datos.

**Sección 2 — Variables geográficas:** `DEPARTAMENTO`, `MUNICIPIO`, `DISTRITO`, `DEPARTAMENTAL`.


In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 80)

RUTA_CRUDO = "../data/raw/MINEDUC_establecimientos_diversificado.csv"

# Se carga todo como texto (dtype=str) a proposito: en el diagnostico queremos ver
# los datos tal como vienen, sin que pandas infiera/convierta tipos por nosotros.
# encoding="utf-8-sig" absorbe el BOM que trae el archivo exportado.
df = pd.read_csv(RUTA_CRUDO, dtype=str, encoding="utf-8-sig")
df.head()

,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,00-01-0158-46,01-01-0026,CIUDAD CAPITAL,ZONA 1,COLEGIO TECNICO PROGRESISTA CETECPRO,2A. AVENIDA 3-59,22512759,LUDIBEL AMARILIS GALICIA DE FLORES,IRMA AMPARO TOLEDO HERNANDEZ,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,DOBLE,DIARIO(REGULAR),GUATEMALA NORTE
1,00-01-0160-46,NaN,CIUDAD CAPITAL,ZONA 1,INSTITUTO DE EDUCACION DIVERSIFICADA 'CENTRO DE ESTUDIOS MERCADOLOGICOS Y PU...,8A AVE. 3-50,22329819,NaN,NaN,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
2,00-01-0162-46,01-102,CIUDAD CAPITAL,ZONA 1,INSTITUTO PRIVADO MIXTO DE EDUCACION DIVERSIFICADA CENTRO EDUCACIONAL GUATEM...,11 CALLE 9-33,22381718,ZIZI ARELY LOPEZ CHINCHILLA,----,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
3,00-01-0168-46,NaN,CIUDAD CAPITAL,ZONA 1,INSTITUTO DE EDUCACION DIVERSIFICADA 'LICEO MARIANO GALVEZ',2A AVE. 9-82,NaN,NaN,NaN,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),GUATEMALA NORTE
4,00-01-0173-46,01-01-0026,CIUDAD CAPITAL,ZONA 1,INSTITUTO NORMAL PARA SEÑORITAS CENTRO AMERICA,1A CALLE 2-64,22323424,LUDIBEL AMARILIS GALICIA DE FLORES,INGRID MARIELA ESPINA ORELLANA,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE


## 1. Descripción global del conjunto de datos

### 1.1 Número de registros y variables

In [2]:
print(f"Registros (filas):    {df.shape[0]:,}")
print(f"Variables (columnas): {df.shape[1]}")
print(f"Columnas: {list(df.columns)}")

Registros (filas):    11,867
Variables (columnas): 17
Columnas: ['CODIGO', 'DISTRITO', 'DEPARTAMENTO', 'MUNICIPIO', 'ESTABLECIMIENTO', 'DIRECCION', 'TELEFONO', 'SUPERVISOR', 'DIRECTOR', 'NIVEL', 'SECTOR', 'AREA', 'STATUS', 'MODALIDAD', 'JORNADA', 'PLAN', 'DEPARTAMENTAL']


### 1.2 Tipo de dato de cada variable

En el archivo crudo **todas** las columnas llegan como texto (`object`). La tabla
compara el tipo con el que llega cada variable contra el tipo que *debería* tener
para el análisis — esa brecha es parte del trabajo de limpieza.

In [3]:
tipo_esperado = {
    "CODIGO":          "str (codigo NN-NN-NNNN-NN)",
    "DISTRITO":        "str (codigo de distrito escolar)",
    "DEPARTAMENTO":    "categorica (catalogo de 22 departamentos)",
    "MUNICIPIO":       "categorica (catalogo de 340 municipios)",
    "ESTABLECIMIENTO": "str (nombre propio)",
    "DIRECCION":       "str (direccion postal)",
    "TELEFONO":        "str (8 digitos, puede haber varios)",
    "SUPERVISOR":      "str (nombre propio)",
    "DIRECTOR":        "str (nombre propio)",
    "NIVEL":           "categorica (constante: DIVERSIFICADO)",
    "SECTOR":          "categorica",
    "AREA":            "categorica",
    "STATUS":          "categorica",
    "MODALIDAD":       "categorica",
    "JORNADA":         "categorica",
    "PLAN":            "categorica",
    "DEPARTAMENTAL":   "categorica (direccion departamental del MINEDUC)",
}
tabla_tipos = pd.DataFrame({
    "tipo_crudo": df.dtypes.astype(str),
    "tipo_esperado": pd.Series(tipo_esperado),
})
tabla_tipos

,tipo_crudo,tipo_esperado
CODIGO,object,str (codigo NN-NN-NNNN-NN)
DISTRITO,object,str (codigo de distrito escolar)
DEPARTAMENTO,object,categorica (catalogo de 22 departamentos)
MUNICIPIO,object,categorica (catalogo de 340 municipios)
ESTABLECIMIENTO,object,str (nombre propio)
DIRECCION,object,str (direccion postal)
TELEFONO,object,"str (8 digitos, puede haber varios)"
SUPERVISOR,object,str (nombre propio)
DIRECTOR,object,str (nombre propio)
NIVEL,object,categorica (constante: DIVERSIFICADO)


### 1.3 Valores faltantes por variable (cantidad y porcentaje)

Se cuentan dos cosas por separado:

- **`NaN` reales**: celdas vacías en el CSV.
- **Marcadores de nulo escritos como texto** (`----`, `-`, `.`, `N/A`, `NULL`,
  `SIN DATO`, `0`, etc.): celdas que *parecen* tener dato pero no informan nada.
  Estos son faltantes disfrazados y deben unificarse a `NaN` en la limpieza.

In [4]:
MARCADORES_NULO = re.compile(r"^\s*(-+|\.+|_+|n/?a|null|nulo|sin\s*dato|ninguno|no\s*tiene|xx+|0)\s*$", re.IGNORECASE)

def contar_marcadores(s):
    return s.dropna().str.match(MARCADORES_NULO).sum()

n = len(df)
faltantes = pd.DataFrame({
    "nan_reales": df.isna().sum(),
    "marcadores_nulo": df.apply(contar_marcadores),
})
faltantes["total_faltante"] = faltantes.sum(axis=1)
faltantes["pct_faltante"] = (faltantes["total_faltante"] / n * 100).round(2)
faltantes.sort_values("total_faltante", ascending=False)

,nan_reales,marcadores_nulo,total_faltante,pct_faltante
DIRECTOR,1732,411,2143,18.06
TELEFONO,946,0,946,7.97
SUPERVISOR,535,0,535,4.51
DISTRITO,532,0,532,4.48
DIRECCION,76,11,87,0.73
ESTABLECIMIENTO,5,0,5,0.04
AREA,0,0,0,0.00
PLAN,0,0,0,0.00
JORNADA,0,0,0,0.00
MODALIDAD,0,0,0,0.00


In [5]:
celdas = df.size
tot = int(faltantes["total_faltante"].sum())
print(f"Celdas totales:            {celdas:,}")
print(f"Celdas faltantes (NaN + marcadores): {tot:,} ({tot/celdas*100:.2f}% del total)")
print(f"Variables con al menos un faltante:  {(faltantes['total_faltante'] > 0).sum()} de {df.shape[1]}")

Celdas totales:            201,739
Celdas faltantes (NaN + marcadores): 4,248 (2.11% del total)
Variables con al menos un faltante:  6 de 17


### 1.4 Cantidad de valores únicos por variable

In [6]:
pd.DataFrame({
    "valores_unicos": df.nunique(),
    "ejemplos": [", ".join(map(str, df[c].dropna().unique()[:3])) for c in df.columns],
})

,valores_unicos,ejemplos
CODIGO,11867,"00-01-0158-46, 00-01-0160-46, 00-01-0162-46"
DISTRITO,1681,"01-01-0026, 01-102, 01-01-0035"
DEPARTAMENTO,23,"CIUDAD CAPITAL, GUATEMALA, EL PROGRESO"
MUNICIPIO,352,"ZONA 1, ZONA 2, ZONA 3"
ESTABLECIMIENTO,6312,"COLEGIO TECNICO PROGRESISTA CETECPRO, INSTITUTO DE EDUCACION DIVERSIFICADA '..."
DIRECCION,7438,"2A. AVENIDA 3-59, 8A AVE. 3-50, 11 CALLE 9-33"
TELEFONO,6571,"22512759, 22329819, 22381718"
SUPERVISOR,1280,"LUDIBEL AMARILIS GALICIA DE FLORES, ZIZI ARELY LOPEZ CHINCHILLA, NORMA ARACE..."
DIRECTOR,5518,"IRMA AMPARO TOLEDO HERNANDEZ, ----, INGRID MARIELA ESPINA ORELLANA"
NIVEL,1,DIVERSIFICADO


### 1.5 Registros duplicados exactos

In [7]:
dup_exactos = df.duplicated().sum()
dup_codigo = df["CODIGO"].duplicated().sum()
print(f"Filas duplicadas exactas (todas las columnas): {dup_exactos}")
print(f"Valores duplicados de CODIGO (posible llave primaria): {dup_codigo}")

Filas duplicadas exactas (todas las columnas): 0
Valores duplicados de CODIGO (posible llave primaria): 0


**Hallazgo:** no hay duplicados exactos y `CODIGO` no se repite, por lo que puede
servir de llave primaria. Los **duplicados parciales** (mismo establecimiento
escrito distinto) se diagnostican por similitud de cadenas en la sección de
`ESTABLECIMIENTO` + `DIRECCION` (P2).

### 1.6 Resumen del estado global

| Métrica | Valor crudo |
|---|---|
| Registros | 11,867 |
| Variables | 17 |
| Tipo de dato | todas llegan como texto (`object`) |
| Duplicados exactos | 0 |
| `CODIGO` duplicados | 0 |
| Variables con faltantes (NaN o marcadores) | ver tabla 1.3 |

Los problemas de calidad, entonces, no están en duplicados exactos sino en:
faltantes disfrazados de texto, formatos inconsistentes, y dominios de
categorías — lo que se diagnostica variable por variable en las secciones
siguientes.

---

## 2. Diagnóstico de variables geográficas

Variables: `DEPARTAMENTO`, `MUNICIPIO`, `DISTRITO`, `DEPARTAMENTAL`.

### 2.1 `DEPARTAMENTO` — dominio y consistencia

Guatemala tiene un catálogo oficial de **22 departamentos**. Se compara el
contenido de la columna contra ese catálogo.

In [8]:
CATALOGO_DEPARTAMENTOS = [
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA", "EL PROGRESO",
    "ESCUINTLA", "GUATEMALA", "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA",
    "PETÉN", "QUETZALTENANGO", "QUICHÉ", "RETALHULEU", "SACATEPÉQUEZ",
    "SAN MARCOS", "SANTA ROSA", "SOLOLÁ", "SUCHITEPÉQUEZ", "TOTONICAPÁN", "ZACAPA",
]

def sin_tildes(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

conteo_depto = df["DEPARTAMENTO"].value_counts(dropna=False).rename("registros").to_frame()
conteo_depto["en_catalogo_oficial"] = conteo_depto.index.isin(CATALOGO_DEPARTAMENTOS)
conteo_depto["en_catalogo_sin_tildes"] = conteo_depto.index.isin([sin_tildes(d) for d in CATALOGO_DEPARTAMENTOS])
conteo_depto

,registros,en_catalogo_oficial,en_catalogo_sin_tildes
DEPARTAMENTO,,,
CIUDAD CAPITAL,2161,False,False
GUATEMALA,1908,True,True
SAN MARCOS,724,True,True
ESCUINTLA,708,True,True
HUEHUETENANGO,591,True,True
QUETZALTENANGO,551,True,True
PETEN,516,False,True
ALTA VERAPAZ,475,True,True
SUCHITEPEQUEZ,437,False,True


In [9]:
print(f"Valores distintos en DEPARTAMENTO: {df['DEPARTAMENTO'].nunique()} (catalogo oficial: 22)")
fuera = conteo_depto[~conteo_depto["en_catalogo_oficial"]]
print(f"Valores que NO coinciden exactamente con el catalogo oficial: {len(fuera)}")
print(list(fuera.index))

Valores distintos en DEPARTAMENTO: 23 (catalogo oficial: 22)
Valores que NO coinciden exactamente con el catalogo oficial: 7
['CIUDAD CAPITAL', 'PETEN', 'SUCHITEPEQUEZ', 'SACATEPEQUEZ', 'QUICHE', 'SOLOLA', 'TOTONICAPAN']


**Hallazgos en `DEPARTAMENTO`:**

1. **Valor fuera de dominio:** `CIUDAD CAPITAL` (2,161 registros) no es un
   departamento; es la forma en que la fuente separa el municipio de Guatemala
   (capital) del resto del departamento `GUATEMALA`. Son 23 valores donde el
   catálogo oficial tiene 22.
2. **Tildes ausentes en toda la columna:** `PETEN`, `QUICHE`, `SOLOLA`,
   `SACATEPEQUEZ`, `SUCHITEPEQUEZ`, `TOTONICAPAN` vienen sin tilde. Ninguno
   coincide exactamente con el catálogo oficial, pero todos coinciden al
   ignorar tildes → es un problema de **formato**, no de dominio.
3. Sin faltantes, sin espacios extra, todo en mayúsculas consistentes (se
   verifica en 2.5).

### 2.2 `MUNICIPIO` — dominio y consistencia

In [10]:
print(f"Valores distintos en MUNICIPIO: {df['MUNICIPIO'].nunique()}")

es_zona = df["MUNICIPIO"].str.match(r"^ZONA \d+$", na=False)
print(f"Registros cuyo MUNICIPIO es una 'ZONA N': {es_zona.sum():,}")
print(f"Zonas distintas: {df.loc[es_zona, 'MUNICIPIO'].nunique()}")
print(f"Municipios 'reales' distintos (sin zonas): {df.loc[~es_zona, 'MUNICIPIO'].nunique()}")

# Las zonas aparecen solo cuando DEPARTAMENTO == CIUDAD CAPITAL?
pd.crosstab(df["DEPARTAMENTO"].eq("CIUDAD CAPITAL"), es_zona,
            rownames=["DEPARTAMENTO == CIUDAD CAPITAL"], colnames=["MUNICIPIO es ZONA N"])

Valores distintos en MUNICIPIO: 352
Registros cuyo MUNICIPIO es una 'ZONA N': 2,161
Zonas distintas: 22
Municipios 'reales' distintos (sin zonas): 330


MUNICIPIO es ZONA N,False,True
DEPARTAMENTO == CIUDAD CAPITAL,,
False,9706,0
True,0,2161


In [11]:
# Municipios por departamento (los 22 oficiales suman 340 municipios en total;
# aqui solo aparecen los que tienen establecimientos de diversificado)
resumen_mun = (df.groupby("DEPARTAMENTO")["MUNICIPIO"]
                 .nunique().sort_values(ascending=False)
                 .rename("municipios_distintos").to_frame())
resumen_mun

,municipios_distintos
DEPARTAMENTO,
HUEHUETENANGO,33
SAN MARCOS,30
QUETZALTENANGO,24
CIUDAD CAPITAL,22
QUICHE,21
SUCHITEPEQUEZ,20
ALTA VERAPAZ,17
JUTIAPA,17
GUATEMALA,17


In [12]:
# Problemas de formato en nombres de municipio
mun = df["MUNICIPIO"].dropna()
chequeos_mun = pd.Series({
    "espacios al inicio/final": (mun != mun.str.strip()).sum(),
    "espacios dobles": mun.str.contains("  ").sum(),
    "minusculas presentes": (mun != mun.str.upper()).sum(),
    "con tilde": mun.str.contains(r"[ÁÉÍÓÚ]", regex=True).sum(),
    "con caracteres no alfabeticos (excepto espacio, punto y numero de zona)":
        mun.str.contains(r"[^A-ZÑÁÉÍÓÚÜ0-9 .]", regex=True).sum(),
}, name="registros_afectados").to_frame()
chequeos_mun

,registros_afectados
espacios al inicio/final,0
espacios dobles,0
minusculas presentes,0
con tilde,0
"con caracteres no alfabeticos (excepto espacio, punto y numero de zona)",0


**Hallazgos en `MUNICIPIO`:**

1. **Mezcla de dos dominios en una sola columna:** para `CIUDAD CAPITAL` la
   columna no trae municipios sino **zonas de la ciudad** (`ZONA 1` … `ZONA 25`,
   2,161 registros). El municipio real de todos esos registros es `GUATEMALA`.
   La tabla cruzada confirma que las zonas aparecen *solo* bajo `CIUDAD CAPITAL`
   y viceversa.
2. **Sin tildes**, igual que `DEPARTAMENTO` (ej. `COBAN`, `AMATITLAN`,
   `PETEN` en nombres de municipio). La validación contra el catálogo oficial
   de 340 municipios (INE) debe hacerse ignorando tildes; la restitución de la
   forma oficial se decide en el plan de limpieza.
3. Sin espacios extra ni minúsculas — el formato general es consistente.
4. Nombres de municipio se repiten entre departamentos (p. ej. hay varios
   `SAN ANTONIO ...`): el municipio solo es identificable junto con su
   departamento, nunca solo.

### 2.3 `DISTRITO` — faltantes y formatos

In [13]:
dis = df["DISTRITO"]
print(f"Faltantes: {dis.isna().sum()} ({dis.isna().mean()*100:.2f}%)")
print(f"Valores distintos: {dis.nunique()}")

# Patron estructural: se reemplaza cada digito por 'N' para ver los formatos presentes
patrones = (dis.dropna()
              .str.replace(r"\d", "N", regex=True)
              .value_counts().rename("registros").to_frame())
patrones["ejemplo"] = [dis.dropna()[dis.dropna().str.replace(r"\d", "N", regex=True) == p].iloc[0]
                       for p in patrones.index]
patrones

Faltantes: 532 (4.48%)
Valores distintos: 1681


,registros,ejemplo
DISTRITO,,
NN-NN-NNNN,6226,01-01-0026
NN-NNN,5039,01-102
NN-,70,01-


In [14]:
# Los 'NN-' son codigos truncados: solo traen el prefijo del departamento
truncados = dis.dropna()[dis.dropna().str.match(r"^\d{2}-$")]
print(f"Codigos truncados ('NN-'): {len(truncados)}")
print(truncados.value_counts().to_string())
print()
# En que departamentos pasa?
df.loc[truncados.index, "DEPARTAMENTO"].value_counts()

Codigos truncados ('NN-'): 70
DISTRITO
01-    67
17-     2
10-     1



DEPARTAMENTO
CIUDAD CAPITAL    54
GUATEMALA         13
PETEN              2
SUCHITEPEQUEZ      1
Name: count, dtype: int64

**Hallazgos en `DISTRITO`:**

1. **532 faltantes (4.48%)** — la única variable geográfica con `NaN` reales.
2. **Tres formatos conviven** en la misma columna:
   - `NN-NN-NNNN` (6,226 registros) — formato largo,
   - `NN-NNN` (5,039 registros) — formato corto,
   - `NN-` (70 registros) — **códigos truncados** que solo conservan el prefijo
     departamental (`16-`, `01-`, ...). Funcionalmente son casi-faltantes.
3. No hay un formato único documentado en la fuente; la limpieza debe decidir
   si se estandariza a un solo formato o si la columna se conserva tal cual
   con su formato de origen documentado en el libro de códigos.

### 2.4 `DEPARTAMENTAL` — dominio y consistencia cruzada

`DEPARTAMENTAL` es la dirección departamental del MINEDUC que administra el
establecimiento. No es lo mismo que `DEPARTAMENTO`, pero deben ser coherentes
entre sí.

In [15]:
conteo_deptal = df["DEPARTAMENTAL"].value_counts(dropna=False).rename("registros").to_frame()
print(f"Valores distintos: {df['DEPARTAMENTAL'].nunique()}")
conteo_deptal

Valores distintos: 26


,registros
DEPARTAMENTAL,
GUATEMALA NORTE,1489
GUATEMALA OCCIDENTE,1035
GUATEMALA SUR,1021
SAN MARCOS,724
ESCUINTLA,708
HUEHUETENANGO,591
QUETZALTENANGO,551
GUATEMALA ORIENTE,524
PETÉN,516


In [16]:
# Consistencia cruzada DEPARTAMENTO <-> DEPARTAMENTAL:
# la departamental deberia "contener" al departamento (ignorando tildes)
depto_norm = df["DEPARTAMENTO"].map(sin_tildes)
deptal_norm = df["DEPARTAMENTAL"].map(sin_tildes)

# CIUDAD CAPITAL pertenece al departamento de Guatemala -> se espera GUATEMALA *
depto_equiv = depto_norm.replace({"CIUDAD CAPITAL": "GUATEMALA"})
coherente = [d in t for d, t in zip(depto_equiv, deptal_norm)]
print(f"Registros donde DEPARTAMENTAL no corresponde al DEPARTAMENTO: {(~np.array(coherente)).sum()}")

# Mapa: que departamentales atienden cada departamento
mapa = (df.assign(DEPTO_EQUIV=depto_equiv)
          .groupby("DEPTO_EQUIV")["DEPARTAMENTAL"]
          .agg(lambda s: sorted(s.unique())).rename("departamentales").to_frame())
mapa[mapa["departamentales"].str.len() > 1]

Registros donde DEPARTAMENTAL no corresponde al DEPARTAMENTO: 0


,departamentales
DEPTO_EQUIV,
GUATEMALA,"[GUATEMALA NORTE, GUATEMALA OCCIDENTE, GUATEMALA ORIENTE, GUATEMALA SUR]"
QUICHE,"[QUICHÉ, QUICHÉ NORTE]"


**Hallazgos en `DEPARTAMENTAL`:**

1. **26 valores** para 22 departamentos: `GUATEMALA` se subdivide en 4
   direcciones (`NORTE`, `SUR`, `OCCIDENTE`, `ORIENTE`) y `QUICHÉ` en 2
   (`QUICHÉ`, `QUICHÉ NORTE`). Es una subdivisión administrativa real del
   MINEDUC, **no** un error de datos.
2. **Inconsistencia de formato con `DEPARTAMENTO`:** aquí los nombres **sí
   traen tilde** (`PETÉN`, `QUICHÉ`, `SOLOLÁ`, `SACATEPÉQUEZ`...) mientras que
   en `DEPARTAMENTO` no. El mismo concepto está escrito de dos maneras en el
   mismo dataset.
3. La consistencia cruzada `DEPARTAMENTO` ↔ `DEPARTAMENTAL` se cumple en el
   100% de los registros (0 contradicciones): buena noticia para usarla de
   verificación en la limpieza.
4. Sin faltantes.

### 2.5 Chequeos de formato transversales (las 4 variables)

In [17]:
geo = ["DEPARTAMENTO", "MUNICIPIO", "DISTRITO", "DEPARTAMENTAL"]
filas = {}
for c in geo:
    s = df[c].dropna()
    filas[c] = {
        "espacios inicio/fin": int((s != s.str.strip()).sum()),
        "espacios dobles": int(s.str.contains("  ").sum()),
        "minusculas": int((s != s.str.upper()).sum()),
        "caracteres invisibles (tab/nbsp)": int(s.str.contains("[\t\u00a0\u200b]").sum()),
        "con tildes": int(s.str.contains(r"[ÁÉÍÓÚáéíóú]", regex=True).sum()),
    }
pd.DataFrame(filas).T

,espacios inicio/fin,espacios dobles,minusculas,caracteres invisibles (tab/nbsp),con tildes
DEPARTAMENTO,0,0,0,0,0
MUNICIPIO,0,0,0,0,0
DISTRITO,0,0,0,0,0
DEPARTAMENTAL,0,0,0,0,2025


### 2.6 Síntesis del diagnóstico geográfico

| Variable | Problema detectado | Registros afectados | Tipo de problema |
|---|---|---|---|
| `DEPARTAMENTO` | `CIUDAD CAPITAL` no pertenece al catálogo oficial de 22 departamentos | 2,161 | dominio |
| `DEPARTAMENTO` | Nombres sin tilde (no coinciden textualmente con el catálogo oficial) | 6 valores / 2,025 registros | formato |
| `MUNICIPIO` | Contiene zonas de la capital (`ZONA 1`–`ZONA 25`) en lugar de municipios | 2,161 | dominio mezclado |
| `MUNICIPIO` | Nombres sin tilde respecto al catálogo oficial de municipios | mayoría de valores | formato |
| `DISTRITO` | Valores faltantes | 532 (4.48%) | faltantes |
| `DISTRITO` | Tres formatos distintos conviviendo (`NN-NN-NNNN`, `NN-NNN`, `NN-`) | 11,335 | formato |
| `DISTRITO` | Códigos truncados `NN-` (solo prefijo departamental) | 70 | casi-faltante |
| `DEPARTAMENTAL` | Escribe los departamentos **con** tilde mientras `DEPARTAMENTO` los escribe **sin** tilde | toda la columna | inconsistencia entre variables |
| `DEPARTAMENTAL` | 26 categorías para 22 departamentos (subdivisiones GUATEMALA×4, QUICHÉ×2) | — | documentar (no es error) |

**Positivo:** sin duplicados exactos, `CODIGO` único, sin espacios extra ni
problemas de mayúsculas en las 4 variables, y coherencia
`DEPARTAMENTO`↔`DEPARTAMENTAL` del 100%.

El plan de limpieza de estas variables (reglas, justificación y riesgos) se
desarrolla en el siguiente entregable.
